# US x1.1 — complete reproducible backtest audit

Research only; `trade_ready=false`. This notebook binds to provider artifact `8831837784` and Experiment 007 artifact `8831960659`, excludes 2026H1, and fails closed on any identity or economics mismatch. It exposes daily scores/ranks, actual rebalance signals, BUY/SELL/HOLD transitions, holdings, costs, period returns, drawdowns and return attribution.

In [ ]:
from __future__ import annotations
import json, os, subprocess, sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
REPO_ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/research').is_dir())
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))
from src.research.us_x1_1_notebook_audit import build_complete_backtest
pd.set_option('display.max_columns',50); pd.set_option('display.width',180); pd.set_option('display.float_format',lambda x:f'{x:,.6f}')

## 1. Frozen inputs and optional full refit

Default mode audits the exact Experiment 007 ledgers. Set `US_X1_1_REFIT=1` to fit US x1.1 twice on the frozen Qlib provider, verify deterministic score/rank/selection identities, and then use the new ledgers.

In [ ]:
PROVIDER_ROOT=Path(os.getenv('US_X1_1_PROVIDER_ROOT','artifacts/source/provider/provider_a')).resolve()
REPRODUCTION_ROOT=Path(os.getenv('US_X1_1_REPRO_ROOT','artifacts/source/reproduction/evidence/us_x1_1_deterministic_reproduction_v1')).resolve()
OUTPUT_DIR=Path(os.getenv('US_X1_1_OUTPUT_DIR','artifacts/evidence/us_x1_1_complete_backtest_notebook')).resolve()
if os.getenv('US_X1_1_REFIT','0')=='1':
    refit=OUTPUT_DIR/'refit/us_x1_1_deterministic_reproduction_v1'
    subprocess.run(['uv','run','python','scripts/run_us_x1_1_deterministic_reproduction.py','--root','.','--provider-uri',str(PROVIDER_ROOT/'data/providers/us'),'--output-dir',str(refit)],check=True)
    REPRODUCTION_ROOT=refit.resolve()
print({'provider':str(PROVIDER_ROOT),'reproduction':str(REPRODUCTION_ROOT),'output':str(OUTPUT_DIR)})

## 2. Reconstruct and verify the complete backtest

The audit derives `Ref($close,-10)/$close-1`, ranks valid names daily, selects Top-15 equal weight every 10 eligible sessions, applies 20 bps to one-way turnover, and reproduces all four Experiment 007 windows.

In [ ]:
audit=build_complete_backtest(PROVIDER_ROOT,REPRODUCTION_ROOT,OUTPUT_DIR)
assert audit.manifest['decision']=='complete_backtest_reproduced'
assert audit.identity_checks['passed'].all() and audit.reproduction_summary['passed'].all()
display(audit.identity_checks.groupby(['check_type','passed'],as_index=False).size())
display(audit.reproduction_summary)
print(json.dumps(audit.manifest['aggregate_result'],indent=2))

## 3. Window economics and performance path

`gross_selection_return` is before costs; `transaction_cost_drag` bridges gross to net. Formal aggregate figures remain the compounded walk-forward window results.

In [ ]:
display(audit.window_attribution)
p=audit.periods.copy(); p['strategy_equity']=(1+p.net_return).cumprod(); p['qqq_equity']=(1+p.qqq_return).cumprod()
ax=p.plot(x='rebalance_date',y=['strategy_equity','qqq_equity'],figsize=(12,5),title='US x1.1 net equity vs QQQ'); ax.grid(alpha=.25); plt.show()
ax=audit.periods.plot(x='rebalance_date',y='strategy_drawdown',figsize=(12,4),title='Window-local strategy drawdown'); ax.grid(alpha=.25); plt.show()

## 4. Complete daily trading signals

`daily_signals.csv` contains every model score and rank, original Top-15 status, economic eligibility, entry/exit prices and 10-session forward return. Change the inspection variables below to review any date.

In [ ]:
print('daily signal rows',len(audit.daily_signals))
WINDOW_TO_INSPECT='2025H1'
dates=audit.daily_signals.loc[audit.daily_signals.window==WINDOW_TO_INSPECT,'datetime'].drop_duplicates().sort_values().tolist()
SIGNAL_DATE_TO_INSPECT=dates[0]
display(audit.daily_signals.query('window==@WINDOW_TO_INSPECT and datetime==@SIGNAL_DATE_TO_INSPECT').sort_values('rank').head(40))

## 5. Actual rebalance signals, trades and holdings

The portfolio acts only every 10 eligible sessions. `rebalance_signals` shows selected names; `trades` explicitly adds exits and labels BUY/SELL/HOLD/INCREASE/DECREASE; `holdings` contains all 720 holding records.

In [ ]:
display(audit.periods[['window','period_index','rebalance_date','holding_end_date','entered_names','exited_names','retained_names']])
INSPECT_WINDOW='2025H1'; INSPECT_PERIOD=1
display(audit.rebalance_signals.query('window==@INSPECT_WINDOW and period_index==@INSPECT_PERIOD').sort_values('rank'))
display(audit.trades.query('window==@INSPECT_WINDOW and period_index==@INSPECT_PERIOD').sort_values(['action','rank','instrument'],na_position='last'))
print('holdings',len(audit.holdings)); assert len(audit.holdings)==720
display(audit.holdings.head(45))

## 6. Every holding-period return, turnover and cost

Turnover is `0.5 × Σ|target−previous|`; cost is turnover × 20 bps; net return is equal-weight gross minus cost; excess is net minus QQQ.

In [ ]:
cols=['window','period_index','rebalance_date','holding_end_date','gross_return','turnover','transaction_cost','net_return','qqq_return','excess_return','strategy_drawdown']
display(audit.periods[cols])
ax=audit.periods.plot.bar(x='rebalance_date',y='turnover',figsize=(12,4),title='Turnover at every rebalance'); ax.grid(axis='y',alpha=.25); plt.show()

## 7. Return-source decomposition

Security attribution is additive across holding periods: target weight × forward return, less allocated entry/exit trading cost. It reconciles security-level gross contribution and total portfolio cost; portfolio performance itself remains compounded.

In [ ]:
display(audit.security_attribution.head(20))
display(audit.security_attribution.tail(20).sort_values('net_contribution'))
ax=audit.security_attribution.head(15).sort_values('net_contribution').plot.barh(x='instrument',y='net_contribution',figsize=(11,5),title='Top net contributors'); ax.grid(axis='x',alpha=.25); plt.show()
display(audit.window_attribution); display(audit.regime_attribution)

## 8. Exported audit package

The output directory contains the authoritative inspection surfaces: `identity_checks.csv`, `daily_signals.csv`, `rebalance_signals.csv`, `trade_ledger.csv`, `holdings.csv`, `period_returns.csv`, `security_attribution.csv`, `window_attribution.csv`, `regime_attribution.csv`, `reproduction_summary.csv` and `audit_manifest.json`, all SHA-bound in the manifest.

In [ ]:
required={'audit_manifest.json','identity_checks.csv','daily_signals.csv','rebalance_signals.csv','trade_ledger.csv','holdings.csv','period_returns.csv','security_attribution.csv','window_attribution.csv','regime_attribution.csv','reproduction_summary.csv'}
files={p.name for p in OUTPUT_DIR.iterdir()}
assert required<=files
print(json.dumps(audit.manifest,indent=2,sort_keys=True))
print('\n'.join(sorted(files)))

## Interpretation limits

This reproduces deterministic revision-provider evidence and does not replace canonical US x1.1. It is not a brokerage simulator: live slippage, liquidity, taxes, restrictions and order routing remain outside scope. 2026H1 is excluded and no parameter search occurs.